# Unified EV Smart Management System
**Integration of both modules:**
- Module 1: Braking Intention Recognition (Yang et al., 2024)
- Module 2: Battery SOC Estimation via LSTM+GA

This notebook shows the full unified pipeline running both models together.

In [ ]:
import sys
sys.path.append('../..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch

from shared.utils import UnifiedEVPipeline

print('PyTorch:', torch.__version__)
print('Device:', 'cuda' if torch.cuda.is_available() else 'cpu')

## 1. Load Unified Pipeline

In [ ]:
pipeline = UnifiedEVPipeline(
    braking_model_path='../../modules/braking/models/final_multitask_model.pth',
    soc_model_path='../../modules/soc/models/lstm_cnn_attention_soc.pth',
    braking_hp_path='../../modules/braking/models/best_ga_hyperparams.json',
)

## 2. Simulate Three Braking Scenarios

In [ ]:
def make_driving_window(scenario='emergency'):
    t = np.linspace(0, 1, 75)
    if scenario == 'light':
        speed       = 40 - 5 * t
        accel       = -0.5 * t
        brake_pedal = 0.05 + 0.1 * t
    elif scenario == 'normal':
        speed       = 50 - 15 * t
        accel       = -2.5 * t
        brake_pedal = 0.15 + 0.25 * t
    else:  # emergency
        speed       = 70 - 50 * t
        accel       = -8 * t
        brake_pedal = 0.4 + 0.5 * t

    noise = np.random.normal(0, 0.02, (75, 3))
    w = np.stack([speed, accel, np.clip(brake_pedal, 0, 1)], axis=1).astype(np.float32)
    w = w + noise.astype(np.float32)
    mean, std = w.mean(0), w.std(0) + 1e-8
    return (w - mean) / std

def make_battery_window(soc_level=0.6):
    voltage     = (3.0 + soc_level * 1.2) * np.ones(50) + np.random.normal(0, 0.01, 50)
    current     = -1.0 * np.ones(50) + np.random.normal(0, 0.05, 50)
    temperature = 25 + np.random.normal(0, 0.3, 50)
    w = np.stack([voltage, current, temperature], axis=1).astype(np.float32)
    mean, std = w.mean(0), w.std(0) + 1e-8
    return (w - mean) / std

scenarios = ['light', 'normal', 'emergency']
soc_levels = [0.8, 0.5, 0.2]   # different battery levels
results = []

for scenario, soc in zip(scenarios, soc_levels):
    r = pipeline.run(
        driving_window=make_driving_window(scenario),
        battery_window=make_battery_window(soc),
        current_soc=soc,
    )
    results.append(r)
    print(f"\nScenario: {scenario.upper()} | SOC: {soc*100:.0f}%")
    print(f"  Predicted class    : {r['braking']['class']}")
    print(f"  Brake intensity    : {r['braking']['intensity']:.4f}")
    print(f"  Energy recovered   : {r['energy']['recovered_normalised']:.6f}")
    print(f"  SOC estimated      : {r['soc']['estimated']*100:.1f}%")
    print(f"  SOC after regen    : {r['soc']['updated']*100:.1f}%")
    print(f"  System action      : {r['system_action']}")

## 3. Unified System Visualisation

In [ ]:
fig = plt.figure(figsize=(15, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.3)

colors    = ['#2ecc71', '#3498db', '#e74c3c']
scenario_labels = ['Light\nBraking', 'Normal\nBraking', 'Emergency\nBraking']

# Row 1: Braking intensity per scenario
ax1 = fig.add_subplot(gs[0, 0])
intensities = [r['braking']['intensity'] for r in results]
bars = ax1.bar(scenario_labels, intensities, color=colors, edgecolor='black', alpha=0.85)
ax1.set_title('Predicted Brake Intensity', fontweight='bold')
ax1.set_ylabel('Intensity [0-1]')
ax1.set_ylim(0, 1.1)
ax1.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, intensities):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{val:.3f}', ha='center', fontsize=9)

# Row 1: Energy recovered
ax2 = fig.add_subplot(gs[0, 1])
energy = [r['energy']['recovered_normalised'] for r in results]
bars = ax2.bar(scenario_labels, energy, color=colors, edgecolor='black', alpha=0.85)
ax2.set_title('Energy Recovered via Regen', fontweight='bold')
ax2.set_ylabel('Energy (normalised)')
ax2.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, energy):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
             f'{val:.4f}', ha='center', fontsize=9)

# Row 1: SOC before vs after
ax3 = fig.add_subplot(gs[0, 2])
soc_before = [soc_levels[i] for i in range(3)]
soc_after  = [r['soc']['updated'] for r in results]
x = np.arange(3)
ax3.bar(x - 0.2, soc_before, 0.35, label='SOC Before', color='#95a5a6', edgecolor='black')
ax3.bar(x + 0.2, soc_after,  0.35, label='SOC After Regen', color='#f39c12', edgecolor='black')
ax3.set_title('SOC Before vs After Regen', fontweight='bold')
ax3.set_ylabel('SOC [0-1]')
ax3.set_xticks(x)
ax3.set_xticklabels(scenario_labels)
ax3.legend()
ax3.grid(True, alpha=0.3, axis='y')

# Row 2: System flow diagram (text-based)
ax4 = fig.add_subplot(gs[1, :])
ax4.axis('off')
flow_text = (
    'UNIFIED EV SMART MANAGEMENT SYSTEM — Integration Flow\n\n'
    'Driving Signals (speed, accel, brake)\n'
    '         ↓\n'
    '[Module 1] LSTM+CNN+Attention+GA → Braking Class + Intensity\n'
    '         ↓                                    ↓\n'
    '  ADAS Safety Alert          Energy Recovered = Intensity × η_regen\n'
    '                                              ↓\n'
    'Battery Signals (V, I, T) → [Module 2] LSTM+CNN+Attention+GA → SOC Estimate\n'
    '                                              ↓\n'
    '                                   SOC_updated = SOC + ΔE_regen\n'
    '                                              ↓\n'
    '                              EV Controller Action (ADAS + BMS)'
)
ax4.text(0.5, 0.5, flow_text, ha='center', va='center', fontsize=10,
         fontfamily='monospace', transform=ax4.transAxes,
         bbox=dict(boxstyle='round', facecolor='#ecf0f1', alpha=0.8))

plt.suptitle('Unified EV Smart Management System — Results', fontsize=14, fontweight='bold')
plt.savefig('../../assets/img/unified_system_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: assets/img/unified_system_results.png')

## 4. Why This Integration Matters

The key insight connecting both modules:

- **Braking intensity** from Module 1 directly determines how much energy can be recovered
- **SOC estimation** from Module 2 tells the BMS whether to prioritise energy recovery or ADAS safety
- Together they enable **proactive regenerative braking control** — the EV can prepare to capture energy *before* full brake application
- Neither paper addresses this joint problem — our unified system is the novel contribution